# Cleaning Data

### Loading libraries

The libraries used are pandas for data manipulation, numpy for array computations (especially nans).

In [1]:
import pandas as pd
import numpy as np
from numpy import nan
np.set_printoptions(suppress=True) # This suppress scientific notation for numpy printouts

### Loading the dataset

we loaded the csv file into a pandas DataFrame named 'movies'

In [2]:
movies = pd.read_csv('Team6_movies_dataset.csv')

In [3]:
movies.shape

(200, 32)

The dataframe contains 200 rows and 32 columns. 


### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

In [4]:
movies.columns #knowing that we had an extra average column we then...
movies.drop('Average', axis=1, inplace=True) #Decided to remove the column with this code
movies.columns # We print again the columns and see that the average column was removed correctly


Index(['Movie', 'Year', 'Runtime', 'Genre', 'Budget', 'Revenue', 'IMDb',
       'Metacritic', 'Rotten Tomatoes', 'Filmaffinity', 'Personal', 'P1', 'P2',
       'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13',
       'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20'],
      dtype='object')

We then decided to copy the current columns in a list so we can reorder the placement of each

In [5]:
movies = movies[['Movie', 'Year', 'Runtime', 'Genre', 'Budget', 'Revenue', 'IMDb',
       'Metacritic', 'Rotten Tomatoes', 'Filmaffinity', 'Personal',
       'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11',
       'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20']]
#Here is the copy of the columns

column_names = ['Title', 'Year', 'Runtime', 'Genre', 'Budget', 'Revenue', 'Personal', 'IMDb', 'Rotten_Tomatoes', 'Metacritic', 'Filmaffinity', 'P1', 'P2',
       'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13',
       'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20']
#Here is the "correct" order

movies.columns = column_names # Here we applied the new column names to the DataFrame.

Finally, we will set the `Title` column as the indexing column of our dataframe.

In [6]:
movies.set_index('Title', inplace=True) #this command sets the 'Title' column as the index of the DataFrame.

### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged for the time being. Data type can be `str` or `int`. It it isn't, we will recast it later.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue`: Same treatment as that of `Budget`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [7]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, (500) Days of Summer to The Social Network
Data columns (total 30 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             200 non-null    int64  
 1   Runtime          200 non-null    int64  
 2   Genre            200 non-null    object 
 3   Budget           199 non-null    float64
 4   Revenue          199 non-null    object 
 5   Personal         200 non-null    int64  
 6   IMDb             199 non-null    float64
 7   Rotten_Tomatoes  200 non-null    int64  
 8   Metacritic       200 non-null    int64  
 9   Filmaffinity     200 non-null    int64  
 10  P1               181 non-null    float64
 11  P2               174 non-null    float64
 12  P3               179 non-null    float64
 13  P4               181 non-null    float64
 14  P5               187 non-null    float64
 15  P6               188 non-null    float64
 16  P7               179 non-null    

We can appreciate that Revenue has a type object and it should be an int or a float the rest of the columns have a correct type. And also the non-null count per column looks right.

#### `Runtime`

In [8]:
np.sort(movies['Runtime'].unique()) #this prints the Runtime column sorting from lowest to highest and only showing unique values

array([ 81,  83,  86,  87,  88,  89,  90,  91,  92,  94,  95,  97,  98,
        99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
       113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
       126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138,
       139, 140, 141, 142, 146, 147, 148, 149, 150, 151, 152, 153, 154,
       155, 156, 158, 162, 163, 164, 165, 166, 169, 170, 175, 176, 180,
       181, 189, 192, 194, 195])

Here we can see that the runtime values look fine.

#### `Budget` and `Revenue`

In [9]:
np.sort(movies['Budget'].unique())  #this prints the Budget column sorting from lowest to highest and only showing unique values

array([  1. ,   1.2,   1.3,   1.5,   2. ,   2.6,   3. ,   3.3,   3.5,
         4. ,   4.5,   5. ,   6. ,   6.4,   6.5,   7. ,   7.5,   8. ,
         8.5,   9. ,  10. ,  11. ,  13. ,  14. ,  15. ,  16. ,  18. ,
        18.5,  19. ,  19.5,  20. ,  22. ,  23. ,  25. ,  28. ,  30. ,
        31. ,  33. ,  34. ,  36. ,  40. ,  44.5,  45. ,  46. ,  47. ,
        50. ,  52. ,  55. ,  58. ,  60. ,  61. ,  63. ,  65. ,  68. ,
        70. ,  75. ,  76. ,  80. ,  90. ,  92. ,  96. ,  97. , 100. ,
       103. , 108. , 110. , 113. , 115. , 118. , 120. , 125. , 135. ,
       145. , 150. , 160. , 165. , 170. , 175. , 178. , 185. , 190. ,
       200. , 237. , 250. , 291. , 300. , 400. , 426. , 460. ,   nan])

Also here we can see that the budget collumn looks fine. But there is a nan value which we already knew it had because when we looked at the columns info

In [10]:
movies['Revenue'].unique()

array(['61', '54', '85.7', '436.2', '32.8', '350.9', '174.2', '34', '57',
       '47', '2,920', '2,320', '2800', '2052', '227', '381.1', '332',
       '244', '135.9', '1447', '376', '1,009', '1.081', '435.3', '103.2',
       '1,348.30', '329', '41.8', '277', '4.1', '55', '235', '2,900',
       '786.4', '1,300', '18.2', '100', '410', '715', '6.4', '72', '600',
       '46.2', '295.5', '678', '250', '1,140', '503.5', '168', '226',
       '47.1', '723.2', '259.5', '48.8', '800', '321', '858.8', '731.6',
       '87.8', '171.5', '327.7', '447.3', '90.4', '1,670.00', '1,109.70',
       '130', '180', '44.6', '313', '15.3', '70', '472', '619', '380.4',
       '457.7', '546.4', '398.5', '571.1', '791', '694.7', '682.7', '598',
       '65.3', '50.3', '171.6', '172.3', '181', nan, '17', '976', '411.1',
       '258', '200', '623', '104', '65.7', '173', '327', '492', '294',
       '365', '690', '394', '1,910', '775', '649', '902.9', '396', '28.5',
       '76.2', '550', '216.7', '78.4', '10.3', '772'

Here in Revenue column we can notice that there are numbers surpasing 3 digits that contain a comma like '1,910', so we need to remove commas. And most importat we knew that Revenue was an object type, so we'll change it to numeric.

In [11]:
movies['Revenue']=movies['Revenue'].str.replace(',', '') # This replaces the commas with a white space
movies['Revenue']=pd.to_numeric(movies['Revenue'], errors='coerce') # conversion to numeric, coercing any remaining errors to NaN.
np.sort(movies['Revenue'].unique())

array([   1.081,    4.1  ,    6.4  ,    7.4  ,    7.5  ,    8.6  ,
         10.3  ,   11.   ,   12.   ,   15.3  ,   17.   ,   18.1  ,
         18.2  ,   20.   ,   28.5  ,   29.   ,   30.   ,   32.8  ,
         33.   ,   34.   ,   37.   ,   40.   ,   41.8  ,   43.   ,
         44.6  ,   45.   ,   46.   ,   46.2  ,   47.   ,   47.1  ,
         48.8  ,   49.   ,   49.2  ,   50.   ,   50.3  ,   51.5  ,
         54.   ,   55.   ,   57.   ,   60.   ,   61.   ,   65.3  ,
         65.7  ,   70.   ,   72.   ,   76.2  ,   78.4  ,   79.   ,
         81.   ,   82.   ,   83.   ,   85.   ,   85.7  ,   86.   ,
         87.8  ,   90.4  ,   92.   ,  100.   ,  103.2  ,  104.   ,
        109.7  ,  119.   ,  120.   ,  122.   ,  123.7  ,  130.   ,
        133.   ,  133.4  ,  135.9  ,  157.   ,  168.   ,  171.5  ,
        171.6  ,  172.3  ,  173.   ,  174.   ,  174.2  ,  180.   ,
        181.   ,  183.   ,  184.   ,  185.   ,  187.   ,  195.2  ,
        200.   ,  216.7  ,  217.   ,  220.   ,  225.   ,  226.

We can see that now the column looks fine, but there is a movie that had a very little revenue, we need to check if this is correct by searching for the movie by the revenue:

In [12]:
movies[(movies['Revenue'] >= 1.0) & (movies['Revenue'] <= 1.9)]

,Year,Runtime,Genre,Budget,Revenue,Personal,IMDb,Rotten_Tomatoes,Metacritic,Filmaffinity,...,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20
Title,,,,,,,,,,,,,,,,,,,,,
Batman Dark Knight Rises,2012,165,Action,250.0,1.081,84,78.0,87,75,92,...,6.0,76.0,NaN,79.0,55.0,88.0,97.0,70.0,88.0,47.0


We can see that the movie Batman Dark Knight Rises has a revenue of $1,081,000 which is not right... and according to **Box Office Mojo**, the revenue for this movie is \$1,081,000,000 dollars so we will replace the given range with 1081.

In [13]:
movies.at['Batman Dark Knight Rises','Revenue'] = 1081. # replaces the old revenue to 1081. for the movie

Now, we clean the rest of the column

In [14]:
# We convert the column back to a string type to remove extra spaces if there are
movies['Revenue'] = movies['Revenue'].astype(str)
movies['Revenue'] = movies['Revenue'].str.replace(' ', '', regex=False) # Remove spaces too, just in case
# And finally convert it back to numeric and print to check if everything is right
movies['Revenue']=pd.to_numeric(movies['Revenue'], errors='coerce')
np.sort(movies['Revenue'].unique())

array([   4.1,    6.4,    7.4,    7.5,    8.6,   10.3,   11. ,   12. ,
         15.3,   17. ,   18.1,   18.2,   20. ,   28.5,   29. ,   30. ,
         32.8,   33. ,   34. ,   37. ,   40. ,   41.8,   43. ,   44.6,
         45. ,   46. ,   46.2,   47. ,   47.1,   48.8,   49. ,   49.2,
         50. ,   50.3,   51.5,   54. ,   55. ,   57. ,   60. ,   61. ,
         65.3,   65.7,   70. ,   72. ,   76.2,   78.4,   79. ,   81. ,
         82. ,   83. ,   85. ,   85.7,   86. ,   87.8,   90.4,   92. ,
        100. ,  103.2,  104. ,  109.7,  119. ,  120. ,  122. ,  123.7,
        130. ,  133. ,  133.4,  135.9,  157. ,  168. ,  171.5,  171.6,
        172.3,  173. ,  174. ,  174.2,  180. ,  181. ,  183. ,  184. ,
        185. ,  187. ,  195.2,  200. ,  216.7,  217. ,  220. ,  225. ,
        226. ,  227. ,  232. ,  233.6,  235. ,  244. ,  250. ,  258. ,
        259.5,  264. ,  273. ,  275. ,  277. ,  286. ,  291. ,  294. ,
        295.5,  313. ,  321. ,  322. ,  327. ,  327.7,  329. ,  332. ,
      

#### Ratings

In [15]:
rating_columns=movies.columns[5:] #Basicly remove the first 5 columns for better handeling of the rating columns
for col in rating_columns:
    print(f"{col} = {np.sort(movies[col].unique())}")

Personal = [52 54 61 62 63 64 65 66 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83
 84 85 86 87 88 89 90 91 92 94]
IMDb = [ 33.  46.  48.  49.  51.  54.  55.  56.  57.  58.  59.  60.  61.  62.
  63.  64.  65.  66.  67.  68.  69.  70.  71.  72.  73.  74.  75.  76.
  77.  78.  79.  80.  81.  82.  83.  84.  85.  86.  87.  88.  89.  90.
  91.  92.  93.  94.  95.  96.  98.  99. 100.  nan]
Rotten_Tomatoes = [ 32  45  50  51  54  55  57  60  62  63  66  68  69  71  72  73  76  77
  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95
  96  97  98  99 100]
Metacritic = [43 48 51 55 56 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76
 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99]
Filmaffinity = [  5  10  15  23  24  25  30  32  36  40  45  50  56  59  60  61  65  66
  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86
  87  88  89  90  91  92  93  94  95  96  97  98  99 100]
P1 = [  0.   3.   4.   5.   6.   8.   9.  10.  13.

We suspect that ratings smaller than $5$ are in a scale from $0$ to $5$, and ratings between $5$ and $10$ are in a scale from $0$ to $10$ so we multiply them by the proper factor to bring them up to a scale from $0$ to $100$.

In [16]:
# Prints all the movies that have a rating of 5 or below
for col in rating_columns:
    print(movies[(movies[col]>0)&(movies[col]<=5)][col].sort_values())

Series([], Name: Personal, dtype: int64)
Series([], Name: IMDb, dtype: float64)
Series([], Name: Rotten_Tomatoes, dtype: int64)
Series([], Name: Metacritic, dtype: int64)
Title
Forrest Gump    5
Name: Filmaffinity, dtype: int64
Title
The Godfather    3.0
Drive            3.0
Ratatouille      4.0
Prisoners        4.0
Lincoln          5.0
Name: P1, dtype: float64
Title
Deadpool 2           2.0
Challengers          2.0
Black Swan           4.0
Avengers: Endgame    4.0
Ex Machina           4.0
Name: P2, dtype: float64
Title
The Thing        3.0
The Lion King    3.0
Nop              4.0
Casablanca       5.0
Argo             5.0
Name: P3, dtype: float64
Title
Apocalypse Now      1.0
Am?lie              2.0
John Wick 4         2.0
F1                  3.0
Get Out             4.0
Nosferatu (1922)    5.0
Name: P4, dtype: float64
Title
Pan?s Labyrinth             2.0
The Silence of the Lambs    3.0
Spirited Away               4.0
Anora                       4.0
The Big Lebowski            4.0
Nam

In [17]:
# Prints all the movies that have a rating of 10 or below
for col in rating_columns:
    print(movies[(movies[col]>0)&(movies[col]<=10)][col].sort_values())

Series([], Name: Personal, dtype: int64)
Series([], Name: IMDb, dtype: float64)
Series([], Name: Rotten_Tomatoes, dtype: int64)
Series([], Name: Metacritic, dtype: int64)
Title
Forrest Gump     5
Barbie          10
Name: Filmaffinity, dtype: int64
Title
Drive                  3.0
The Godfather          3.0
Ratatouille            4.0
Prisoners              4.0
Lincoln                5.0
Terminator 1           6.0
Dead Poets Society     8.0
The Menu               9.0
The Thing             10.0
Se7en                 10.0
Name: P1, dtype: float64
Title
Deadpool 2           2.0
Challengers          2.0
Black Swan           4.0
Avengers: Endgame    4.0
Ex Machina           4.0
Pulp fiction         6.0
Life of Pi           9.0
Name: P2, dtype: float64
Title
The Lion King                                  3.0
The Thing                                      3.0
Nop                                            4.0
Casablanca                                     5.0
Argo                               

Looking at the ratings and the title of the movies we decided to aknowledge a mistake in the ratings scale so we'll multiply for the correct factor

In [18]:
# multiplies all movie ratings from 0 to 5 times 20 for a correct scale
for col in rating_columns:
    movies[col]=np.where((movies[col]>0)&(movies[col]<=5), movies[col]*20, movies[col])

In [19]:
# multiplies all movie ratings from 0 to 10 times 10 for a correct scale
for col in rating_columns:
    movies[col]=np.where((movies[col]>0)&(movies[col]<=10), movies[col]*10, movies[col])

#### `Genre`

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [20]:
movies['Genre'].unique()

array(['Romance', 'Terror', 'Comedy', 'Horror', 'Drama', 'Sci-Fi',
       'Action', 'Animation', 'Thriller', 'Western', 'Crime', 'Suspense',
       'War', 'Adventure', 'Mystery', 'Musical', 'Childish',
       'Ciencia ficci«n / Terror', 'Ciencia ficci«n / Acci«n',
       'Biographical', 'Fantasy', 'Sports', 'Historical'], dtype=object)

We can see that SciFi is separated with "-", some double genres separated with "/" and a symbol "«" replacing the ó in spanish genres, so we'll replace them

In [21]:
movies['Genre'] = movies['Genre'].str.replace(' ','') # just in case we didn't notice an space
movies['Genre'] = movies['Genre'].str.replace('-','')
movies['Genre'] = movies['Genre'].str.replace('/','')
movies['Genre'] = movies['Genre'].str.replace('«','on')
movies['Genre'].unique()

array(['Romance', 'Terror', 'Comedy', 'Horror', 'Drama', 'SciFi',
       'Action', 'Animation', 'Thriller', 'Western', 'Crime', 'Suspense',
       'War', 'Adventure', 'Mystery', 'Musical', 'Childish',
       'CienciaficcionnTerror', 'CienciaficcionnAccionn', 'Biographical',
       'Fantasy', 'Sports', 'Historical'], dtype=object)

Now we just need to change the spanish genres, the 'Childish' genre to family and we decided to also change 'Terror' to just 'Horror':

In [22]:
movies['Genre'] = movies['Genre'].replace('Childish', 'Family')
movies['Genre'] = movies['Genre'].replace('CienciaficcionnTerror', 'SciFi')
movies['Genre'] = movies['Genre'].replace('CienciaficcionnAccionn', 'SciFi')
movies['Genre'] = movies['Genre'].replace('Terror', 'Horror')
np.sort(movies['Genre'].astype(str).unique())

array(['Action', 'Adventure', 'Animation', 'Biographical', 'Comedy',
       'Crime', 'Drama', 'Family', 'Fantasy', 'Historical', 'Horror',
       'Musical', 'Mystery', 'Romance', 'SciFi', 'Sports', 'Suspense',
       'Thriller', 'War', 'Western'], dtype=object)

Okey now we have the correct genre names. But we need a final check to see if there is any problem remaining

In [23]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, (500) Days of Summer to The Social Network
Data columns (total 30 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             200 non-null    int64  
 1   Runtime          200 non-null    int64  
 2   Genre            200 non-null    object 
 3   Budget           199 non-null    float64
 4   Revenue          199 non-null    float64
 5   Personal         200 non-null    int64  
 6   IMDb             199 non-null    float64
 7   Rotten_Tomatoes  200 non-null    int64  
 8   Metacritic       200 non-null    int64  
 9   Filmaffinity     200 non-null    int64  
 10  P1               181 non-null    float64
 11  P2               174 non-null    float64
 12  P3               179 non-null    float64
 13  P4               181 non-null    float64
 14  P5               187 non-null    float64
 15  P6               188 non-null    float64
 16  P7               179 non-null    

Types and non-null count looks fine. Now, let's verify that the rating columns have the correct scale

In [24]:
movies.iloc[:,3:11].describe()

,Budget,Revenue,Personal,IMDb,Rotten_Tomatoes,Metacritic,Filmaffinity,P1
count,199.000000,199.000000,200.000000,199.000000,200.000000,200.000000,200.000000,181.000000
mean,73.566834,414.496985,78.655000,77.145729,86.545000,77.025000,80.690000,72.016575
std,86.764121,515.116529,6.153479,12.192226,11.492535,11.374589,15.767912,24.244008
min,1.000000,4.100000,52.000000,33.000000,32.000000,43.000000,15.000000,0.000000
25%,14.000000,80.000000,75.000000,69.000000,81.750000,69.000000,74.000000,55.000000
50%,40.000000,235.000000,79.000000,78.000000,90.000000,77.000000,85.000000,80.000000
75%,109.000000,539.650000,83.000000,86.500000,94.000000,86.000000,91.000000,91.000000
max,460.000000,2920.000000,94.000000,100.000000,100.000000,99.000000,100.000000,100.000000


In [25]:
movies.iloc[:,11:21].describe()

,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11
count,174.000000,179.000000,181.000000,187.000000,188.000000,179.000000,176.000000,176.000000,181.000000,186.000000
mean,71.183908,71.351955,72.143646,70.101604,69.670213,73.195531,71.585227,69.585227,68.955801,72.048387
std,25.528644,25.345348,23.423903,25.193132,25.200114,23.559410,26.450679,24.794322,27.457608,25.179632
min,0.000000,0.000000,11.000000,12.000000,12.000000,13.000000,11.000000,12.000000,0.000000,11.000000
25%,54.250000,60.000000,56.000000,52.500000,52.000000,59.500000,58.750000,50.500000,47.000000,58.250000
50%,81.000000,80.000000,80.000000,80.000000,79.000000,80.000000,81.000000,79.500000,81.000000,80.000000
75%,89.750000,91.000000,90.000000,90.000000,90.000000,93.000000,92.000000,89.000000,90.000000,92.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [26]:
movies.iloc[:,21:].describe()

,P12,P13,P14,P15,P16,P17,P18,P19,P20
count,184.000000,174.000000,181.000000,177.000000,180.000000,185.000000,179.000000,179.000000,186.000000
mean,70.538043,66.264368,70.055249,73.604520,72.538889,68.740541,71.888268,72.709497,71.413978
std,25.370450,29.432380,27.327179,22.381039,26.024776,26.512256,26.510096,24.709423,24.955650
min,0.000000,11.000000,11.000000,12.000000,12.000000,0.000000,0.000000,11.000000,0.000000
25%,54.750000,40.000000,52.000000,60.000000,58.500000,53.000000,56.500000,59.500000,57.000000
50%,80.000000,79.000000,80.000000,81.000000,82.500000,79.000000,80.000000,81.000000,80.000000
75%,89.000000,92.000000,91.000000,90.000000,91.000000,89.000000,91.500000,91.000000,89.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


Everything looks perfect so now we'll sace the clean data set into a csv file named 'Team6_clean'

### Save dataframe to disk

In [27]:
#movies.to_csv('Team6_clean.csv')